In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

[ RAG 구현 절차 ]

    1.	문서의 내용을 읽는다(document_loader를 이용)
    (1)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/ 
    (2)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
    %pip install --upgrade --quiet  docx2txt
    2.	문서를 쪼갠다(한번에 이해하고 처리할 수 있는 입력+출력 토큰수가 제한)
    (1)	 https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/#splitting-text-from-languages-without-word-boundaries 
    %pip install -qU langchain-text-splitters
    3.	쪼갠 문서를 임베딩하여 vector database에 넣음
    (1)	OpenAIEmbeddings나 UpstageEmbeddings이용해서 임베딩
    (2)	https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/  
    %pip install –q langchain-chroma
    4.	질문을 이용해 유사도 검색
    5.	유사도 검색한 문서를 LLM에 질문으로 전달하여 답변 얻음(제공되는 Prompt활용)
    (1)	https://python.langchain.com/v0.2/docs/tutorials/rag/
    %pip install –q langchain langchainhub
    http://smith.langchain.com에서 key생성 .env key(LANGCHAIN_API_KEY) 추가

## 0. 패키지 설치

In [2]:
# 문서 읽어오기
%pip install --quiet  docx2txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 텍스트를 청크로 나누는 기능만 있는 경량 모듈
%pip install -q langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [4]:
# 벡터DB (로컬DB)
%pip install -q langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [5]:
# 제공되는 prompt 사용
%pip install -q langchain langchainhub

Note: you may need to restart the kernel to use updated packages.


## 1. 문서 읽기(X)

In [6]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
document = loader.load()

CPU times: total: 5.61 s
Wall time: 5.84 s


In [7]:
len(document)

1

In [8]:
document[0].page_content[:200]

'소득세법\n\n소득세법\n\n[시행 2025. 7. 1.] [법률 제20615호, 2024. 12. 31., 일부개정]\n\n기획재정부(재산세제과(양도소득세)) 044-215-4312\n\n기획재정부(소득세제과(근로소득)) 044-215-4216\n\n기획재정부(금융세제과(이자소득, 배당소득)) 044-215-4233\n\n기획재정부(소득세제과(사업소득, 기타소득)) 044-2'

## 2. 문서를 쪼개면서 읽기(o)

In [9]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter(  # 문서를 쪼개는 기준이 문자수
    chunk_size=1500, #문서를 쪼갤때 1500글자씩 쪼개
    chunk_overlap=200
)
# 1번째 chunk 1~1450글자
# 2번째 chunk 1250~1750글자
document = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서 쪼개면서 읽는 시간 :', runtime)

문서 쪼개면서 읽는 시간 : 5.169539213180542


In [10]:
# chunk 갯수
len(document)

183

In [11]:
len(document[0].page_content)

1464

In [12]:
# chunk의 글자수들
# [len(doc.page_content) for doc in document]
print(max(len(doc.page_content) for doc in document))
print(min(len(doc.page_content) for doc in document))

1497
1055


## 3. 쪼갠문서를 임베딩 -> 벡터 데이터베이스 저장
- 임베딩 모델 : openAI API의 text-embedding-3-large (기본:text-embedding-ada-002)
- 벡터 데이터베이스 : chroma

In [13]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
# https://python.langchain.com/v0.2/docs/how_to/embed_text/
embedding = OpenAIEmbeddings(
    model = "text-embedding-3-large"
)

In [14]:
embeddings = embedding.embed_documents(
    [
        "소득세법 어쩌구 저쩌구",
        document[0].page_content
    ]
)
len(embeddings), len(embeddings[0])

(2, 3072)

In [15]:
len(embeddings), len(embeddings[0]), len(embeddings[1])

(2, 3072, 3072)

In [16]:
len(embedding.embed_query("소득세"))

3072

In [17]:
%%time
from langchain_chroma import Chroma
# 데이터를 처음 저장할 때
# database = Chroma.from_documents(                                 
#     documents=document,
#     embedding=embedding,
#     collection_name="tax-collection", # 생략시 이름 랜덤
#     persist_directory='./chroma'      # 생략시 로컬데이터베이스에 저장안됨. 프로그램 종료시 db날라감
# )
# 이미 저장된 vector DB를 사용할 때
database = Chroma(
    embedding_function=embedding,
    collection_name="tax-collection",
    persist_directory='./chroma'
)

CPU times: total: 812 ms
Wall time: 1.55 s


## 4. vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [18]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,
                                           k=3) # 기본 k는 4

In [19]:
retrieved_docs

[]

## 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM 전달하여 답변 생성

In [20]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [21]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세 전문자입니다
- [context]를 참고해서 사용자의 질문에 답변해 주세요
[context]는 다음과 같아요
{retrieved_docs}
Question : {query}"""

In [22]:
ai_message = llm.invoke(prompt)

In [23]:
print(ai_message.content)

연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 근로소득세 과세표와 공제액 등을 고려해야 합니다. 아래는 2023년 기준으로 일반적인 근로소득세 계산 방법입니다.

1. 근로소득공제 계산  
연봉 50,000,000원에 대한 근로소득공제는 아래와 같이 계산됩니다.

- 연봉 5,000만원 기준, 근로소득공제액은 약 1,650만원입니다.  
(근로소득공제 공식 참고:  
연봉별 공제액은 일정 표에 따라 결정됩니다. 5,000만원의 경우 대략적으로 1,650만원 정도입니다.)

2. 과세표준 계산  
과세표준 = 연봉 - 근로소득공제  
= 50,000,000 - 16,500,000  
= 33,500,000원

3. 세율 및 누진세 계산  
2023년 기준 세율에 따라 과세표준에 적용됩니다.

- 1,200만원 이하: 6%  
- 1,200만원 초과 ~ 4,600만원 이하: 15% (초과분은 6% + 누진세)  
- 4,600만원 초과: 24% 등

우리 예시에서는 과세표준이 3,350만원이기 때문에:

- 1,200만원까지: 1,200만원 × 6% = 72만원  
- 1,200만원 초과 ~ 3,350만원: (3,350만원 - 1,200만원) = 2,150만원 × 15% = 322.5만원

세액 합계: 72만원 + 322.5만원 = 394.5만원  

4. 주민세 및 기타 세액 공제 고려  
주민세(일반적으로 세액의 10%)가 추가됩니다:  
394.5만원 × 10% = 약 39.45만원

총 세액 약 **433만원** 정도가 될 것으로 예상됩니다.

**참고:**  
이 계산은 기본공제와 표준세율을 반영한 간단 추정입니다. 실제 세액은 공제항목, 보험료, 기타 소득공제 등 다양한 요소에 따라 차이가 있을 수 있으며, 상세 계산을 원하시면 세무사와 상담하시는 것이 좋습니다.


## 5. Augmentation을 위한 제공되는 Prompt활용하여 langchain으로 답변 생성

In [24]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")
prompt

C:\Users\Admin\anaconda3\envs\LLM\lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

### RetrievalQA를 통해 LLM전달 (create_retrieval_chain이 대체)
     query -> retriever전달(백터 검색 수행) 
     -> retrieval문서 -> prompt의 {context}에 삽입
     -> query -> prompt의 {question}에 삽입
    

In [25]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = database.as_retriever(search_kwargs={'k':5}),
    chain_type_kwargs={"prompt":prompt}
)

In [26]:
ai_message = qa_chain.invoke({"query":query})

In [27]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '2023년 한국의 소득세는 누진세 구조로, 연봉 5천만 원에 대해 계산하면 약 1,104만 원 정도입니다. 이는 기본 공제와 인적공제 등을 고려하지 않은 추정치입니다. 정확한 세액은 공제항목과 세법 변경에 따라 달라질 수 있습니다.'}